# ASMR Helper — 无状态云端 GPU 算力节点 / Stateless Cloud GPU Worker (Colab T4 / L4)

**中文说明**：本 Notebook 在免费或 Pro 版 Google Colab（推荐 **T4 16GB** 或 **L4 24GB** 运行时）上一键拉起无状态 `packages.gpu_worker` 推理服务，并通过免注册的 **Cloudflare Quick Tunnel** 暴露临时加密 HTTPS 地址与随机 Bearer Token。本机 Agent 或桌面端只需粘贴输出的连接信息，即可将人声分离（Demucs）、语音识别（Faster-Whisper / Anime-Whisper / FunASR）、强制对齐与本地 TTS 卸载至云端 GPU，输入输出自动经无损 `.flac` 压缩传输（节省约 60% 带宽）。

**English Overview**: This notebook launches the stateless `packages.gpu_worker` FastAPI server on Google Colab (**T4 16GB** or **L4 24GB** GPU runtime recommended) and exposes it via a zero-account **Cloudflare Quick Tunnel** protected by a random Bearer Token. Paste the printed HTTPS URL and token into your local ASMR Helper Agent or configuration to offload vocal separation, ASR, forced alignment, and batch local TTS with automatic lossless `.flac` transport and stage-aware VRAM eviction.

## 步骤 1：挂载 Google Drive 持久化模型权重缓存（可选）/ Step 1: Mount Google Drive for Persistent Model Cache (Optional)

**中文**：开启 `MOUNT_GOOGLE_DRIVE = True` 后，模型权重将保存在 `/content/drive/MyDrive/AsmrHelperModels` 下，自动设置 `ASMR_HELPER_MODEL_ROOT`、`HF_HOME` 与 `MODELSCOPE_CACHE`，下次重启 Colab 会话无需重复下载数 GB 模型权重。

**English**: When `MOUNT_GOOGLE_DRIVE = True`, model weights are stored under `/content/drive/MyDrive/AsmrHelperModels` and wired to `ASMR_HELPER_MODEL_ROOT`, `HF_HOME`, and `MODELSCOPE_CACHE` so weights persist across Colab sessions.

In [ ]:
import os
from pathlib import Path

MOUNT_GOOGLE_DRIVE = True  # 设为 False 则仅使用当前实例临时磁盘 / Set False for ephemeral local storage
DRIVE_CACHE_ROOT = Path("/content/drive/MyDrive/AsmrHelperModels")
FALLBACK_CACHE_ROOT = Path("/content/AsmrHelperModels")

cache_root = FALLBACK_CACHE_ROOT
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive  # type: ignore[import-not-found]

        drive.mount("/content/drive", force_remount=False)
        cache_root = DRIVE_CACHE_ROOT
        print("[Drive] 已成功挂载 Google Drive / Mounted Google Drive successfully.")
    except Exception as exc:
        print(f"[Drive] 未在 Colab 环境或跳过挂载，改用临时目录 / Falling back to ephemeral disk: {exc}")

hf_home = cache_root / "huggingface"
ms_cache = cache_root / "modelscope"
torch_home = cache_root / "torch"
for folder in (cache_root, hf_home, ms_cache, torch_home):
    folder.mkdir(parents=True, exist_ok=True)

os.environ["ASMR_HELPER_MODEL_ROOT"] = str(cache_root)
os.environ["HF_HOME"] = str(hf_home)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(hf_home / "hub")
os.environ["MODELSCOPE_CACHE"] = str(ms_cache)
os.environ["TORCH_HOME"] = str(torch_home)

print(f"ASMR_HELPER_MODEL_ROOT = {os.environ['ASMR_HELPER_MODEL_ROOT']}")
print(f"HF_HOME                = {os.environ['HF_HOME']}")
print(f"MODELSCOPE_CACHE       = {os.environ['MODELSCOPE_CACHE']}")

## 步骤 2：极速环境依赖与仓库准备（适配 T4 / L4）/ Step 2: Fast Dependency & Repository Setup (Colab T4 / L4)

**中文**：直接复用 Colab 预装的 CUDA PyTorch，仅安装轻量推理服务依赖（FastAPI、SoundFile、Faster-Whisper、Demucs、Transformers 等），通常 20–45 秒即可完成准备。

**English**: Reuses Colab's pre-installed CUDA PyTorch and installs only the missing inference packages (`fastapi`, `soundfile`, `faster-whisper`, `demucs`, `transformers`, etc.) for rapid T4/L4 cold start.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = os.environ.get("ASMR_HELPER_REPO_URL", "https://github.com/asmr-helper/AsmrHelper.git")
REPO_BRANCH = os.environ.get("ASMR_HELPER_REPO_BRANCH", "dev")
REPO_DIR = Path("/content/AsmrHelper")

# 1. 检查 GPU 显卡状态 / Verify GPU accelerator (T4 / L4)
try:
    import torch

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"[GPU] 检测到 CUDA 加速卡 / Detected GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    else:
        print("[WARN] 当前未开启 GPU！请在菜单『代码执行程序 -> 更改运行时类型』中选择 T4 或 L4 GPU。")
        print("[WARN] CUDA not available! Please switch runtime type to T4 or L4 GPU.")
except ImportError:
    print("[WARN] PyTorch not found in base image; pip will install required packages.")

# 2. 拉取或更新仓库代码 / Clone or update repository
if not (REPO_DIR / "packages" / "gpu_worker").exists():
    if REPO_DIR.exists():
        print(f"[Repo] 使用已挂载目录 / Using existing directory: {REPO_DIR}")
    else:
        print(f"[Repo] 正在克隆仓库 / Cloning {REPO_URL} ({REPO_BRANCH})...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )
else:
    print(f"[Repo] 仓库已就绪 / Repository ready at {REPO_DIR}")

# 3. 极速安装推理依赖（复用已有 torch）/ Fast dependency install reusing Colab torch
PACKAGES = [
    "fastapi>=0.115.0",
    "uvicorn>=0.34.0",
    "pydantic>=2.0.0,<3.0.0",
    "httpx>=0.25.0",
    "soundfile>=0.12.0",
    "imageio-ffmpeg>=0.4.0",
    "pyyaml>=6.0.0",
    "huggingface-hub>=0.34,<2",
    "faster-whisper>=1.0.0",
    "demucs>=4.0.0",
    "transformers>=4.52.0,<5.0.0",
    "accelerate>=0.20.0",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *PACKAGES],
    check=True,
)

os.environ["PYTHONPATH"] = str(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("[Setup] 依赖与 PYTHONPATH 配置完成！/ Dependencies & PYTHONPATH configured!")

## 步骤 3：启动 GPU Worker 与免注册 Cloudflare Tunnel / Step 3: Launch GPU Worker & Free Cloudflare Quick Tunnel

**中文**：自动生成高强度随机 Bearer Token，在后台启动 `packages.gpu_worker.cli --serve`（端口 `8765`），并拉起免账号 `cloudflared tunnel --url http://127.0.0.1:8765`。看到下方输出的 `https://xxx.trycloudflare.com` 与 Token 后，直接复制发给本地 Agent 即可开始高速推理！

**English**: Generates a random Bearer Token, starts `packages.gpu_worker.cli --serve` on `127.0.0.1:8765` in the background, and launches an account-free `cloudflared tunnel --url http://127.0.0.1:8765`. Copy the printed `https://xxx.trycloudflare.com` URL and Bearer Token directly into your local Agent.

In [ ]:
import os
from pathlib import Path
import re
import secrets
import stat
import subprocess
import sys
import time
import urllib.request

import httpx

WORKER_HOST = "127.0.0.1"
WORKER_PORT = 8765
REPO_DIR = Path("/content/AsmrHelper")
CLOUDFLARED_BIN = Path("/usr/local/bin/cloudflared")
if not os.access(CLOUDFLARED_BIN.parent, os.W_OK):
    CLOUDFLARED_BIN = Path("/tmp/cloudflared")

# 1. 生成随机安全令牌 / Generate random Bearer Token
worker_token = os.environ.get("ASMR_WORKER_TOKEN") or secrets.token_urlsafe(24)
os.environ["ASMR_WORKER_TOKEN"] = worker_token

# 2. 后台拉起 packages.gpu_worker.cli --serve / Start stateless GPU worker in background
worker_env = dict(os.environ)
worker_env["PYTHONPATH"] = str(REPO_DIR)
worker_log = open("/tmp/asmr_gpu_worker.log", "w", encoding="utf-8")
worker_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "packages.gpu_worker.cli",
        "--serve",
        "--host",
        WORKER_HOST,
        "--port",
        str(WORKER_PORT),
        "--token",
        worker_token,
    ],
    cwd=str(REPO_DIR) if REPO_DIR.exists() else None,
    env=worker_env,
    stdout=worker_log,
    stderr=subprocess.STDOUT,
)

# 等待本地端口就绪 / Wait for local FastAPI healthcheck
local_url = f"http://{WORKER_HOST}:{WORKER_PORT}"
caps_json = None
for _ in range(40):
    if worker_proc.poll() is not None:
        worker_log.flush()
        raise RuntimeError(Path("/tmp/asmr_gpu_worker.log").read_text(encoding="utf-8")[-2000:])
    try:
        resp = httpx.get(
            f"{local_url}/v1/capabilities",
            headers={"Authorization": f"Bearer {worker_token}"},
            timeout=2.0,
        )
        if resp.status_code == 200:
            caps_json = resp.json()
            break
    except Exception:
        pass
    time.sleep(0.5)

if caps_json is None:
    raise RuntimeError("GPU Worker failed to become ready on 127.0.0.1:8765")
print(f"[Worker] 本地服务已启动 / Local server ready: {caps_json}")

# 3. 下载并启动免注册 cloudflared Quick Tunnel / Download & start account-free cloudflared
if not CLOUDFLARED_BIN.exists():
    cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    print(f"[Tunnel] 正在下载 cloudflared / Downloading {cf_url}...")
    urllib.request.urlretrieve(cf_url, str(CLOUDFLARED_BIN))
    CLOUDFLARED_BIN.chmod(CLOUDFLARED_BIN.stat().st_mode | stat.S_IEXEC)

tunnel_proc = subprocess.Popen(
    [
        str(CLOUDFLARED_BIN),
        "tunnel",
        "--no-autoupdate",
        "--url",
        "http://127.0.0.1:8765",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
public_url = None
deadline = time.time() + 45.0
while time.time() < deadline and tunnel_proc.stdout is not None:
    line = tunnel_proc.stdout.readline()
    if not line and tunnel_proc.poll() is not None:
        break
    match = tunnel_pattern.search(line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Failed to obtain https://*.trycloudflare.com URL from cloudflared")

banner = "=" * 72
print(f"\n{banner}")
print("ASMR Helper Colab GPU Worker 已上线！/ Remote GPU Worker is LIVE!")
print(banner)
print(f"Public URL   : {public_url}")
print(f"Bearer Token : {worker_token}")
print("-" * 72)
print("复制以下内容发送给本地 Agent（或写入环境变量）/ Paste to your local Agent:")
print(f"ASMR_HELPER_REMOTE_WORKER_URL={public_url}")
print(f"ASMR_HELPER_REMOTE_WORKER_TOKEN={worker_token}")
print(f"{banner}\n")